# `Segment` basics -- building OBC segments standalone & interior

This notebook is a short tour of the `regional_mom6.segment.Segment` class 

The key idea: a `Segment` only needs a small slice of a horizontal grid (and,
optionally, a matching `regional_mom6.Topo` for masking) to describe a single
straight, index-aligned MOM6 open boundary segment. It never needs the full
`regional_mom6.experiment.Experiment` -- so you can build and inspect segments
entirely on their own, without setting up a whole case first.

We'll use a small synthetic grid + flat bathymetry (with a strip of land
along the top) so this notebook runs instantly with no data download.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

from regional_mom6.grid import Grid
from regional_mom6.topo import Topo
from regional_mom6.segment import Segment

## Step 1: A small synthetic grid + bathymetry

A 20x20 T-cell rectilinear domain, flat at 100 m depth, with a full-width strip
of land across the top 5 rows (rows 15-19) -- e.g. a coastline running along
the domain's north edge -- so we have some land to build boundaries around and
against.

In [2]:
grid = Grid(
    resolution=1,
    xstart=0,
    lenx=20,
    ystart=0,
    leny=20,
    name="segment_demo_domain",
    type="rectilinear_cartesian",
)
topo = Topo(grid, min_depth=5.0, git=False)
topo.set_flat(100.0)

depth = topo.depth.values.copy()
depth[15:20, :] = 0.0  # land strip along the north edge, full width
topo.depth = depth

hgrid = grid._supergrid.to_ds(name=grid.name, author="segment-basics-demo")
hgrid

<xarray.Dataset> Size: 79kB
Dimensions:   (nyp: 41, nxp: 41, ny: 40, nx: 40)
Dimensions without coordinates: nyp, nxp, ny, nx
Data variables:
    y         (nyp, nxp) float64 13kB 0.0 0.0 0.0 0.0 ... 20.0 20.0 20.0 20.0
    x         (nyp, nxp) float64 13kB 0.0 0.5 1.0 1.5 ... 18.5 19.0 19.5 20.0
    dy        (ny, nxp) float64 13kB 5.56e+04 5.56e+04 ... 5.56e+04 5.56e+04
    dx        (nyp, nx) float64 13kB 5.56e+04 5.56e+04 ... 5.224e+04 5.224e+04
    area      (ny, nx) float64 13kB 3.091e+09 3.091e+09 ... 2.909e+09 2.909e+09
    angle_dx  (nyp, nxp) float64 13kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes:
    type:       MOM6 supergrid
    grid_type:  rectilinear_cartesian
    name:       segment_demo_domain
    Created:    2026-11-10T15:03:47.792136
    Author:     segment-basics-demo

## Step 2: The easy case -- a full cardinal edge

`Segment.cardinal` is a convenience wrapper for the 4 full-edge cases
(`'north'`, `'south'`, `'east'`, `'west'`). Pass the `hgrid` and (optionally) a
`topo` to get land masking for free.

In [3]:
south_segment = Segment.cardinal(hgrid, "south", "segment_001", topo=topo)

print(south_segment.mom6_obc_position_string())
south_segment.lon.values, south_segment.mask.values

J=0,I=0:N


(array([ 0. ,  0.5,  1. ,  1.5,  2. ,  2.5,  3. ,  3.5,  4. ,  4.5,  5. ,
         5.5,  6. ,  6.5,  7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. , 10.5,
        11. , 11.5, 12. , 12.5, 13. , 13.5, 14. , 14.5, 15. , 15.5, 16. ,
        16.5, 17. , 17.5, 18. , 18.5, 19. , 19.5, 20. ]),
 array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]))

## Step 3: An interior/arbitrary segment

For anything that isn't a full outer edge -- a partial edge, or a truly
interior line -- use `Segment.from_hgrid` directly.

A few things to keep in mind (see the module docstring in `segment.py` for the
full explanation):

- `axis="nyp"` gives a *horizontal* line (varies in x, sits on MOM6's V-point
  grid); `axis="nxp"` gives a *vertical* line (varies in y, U-point grid).
- `index` is a **supergrid** index (`2 * k + 1` for native T-row/column `k`),
  not a native grid index.
- `index_range` must start and end on a cell corner (an even supergrid
  index), since MOM6 reads a segment's data from its first corner -- to open
  T-cells `lo` to `hi`, use `slice(2 * lo, 2 * hi + 3)`.
- `ocean_side` says which compass side is the interior/ocean side -- `Segment`
  handles MOM6's own-position-masking asymmetry internally, so you never have
  to shift `index` yourself to compensate.
- an interior segment's two endpoints must be **land-capped** -- the T-cell
  one step past each end, on the land side, has to actually be land, or
  `from_hgrid` raises `ValueError` (a segment that ends flush against open
  water at a concave corner can blow MOM6 up within its first timestep). Below,
  T-row 14 sits just south of our land strip (which starts at row 15), and the
  land strip is full-width -- wider than the segment's own T-columns 7-15 --
  so both endpoints land safely under it.

Here we cut an interior line along native T-row 14 (supergrid index `29`),
spanning T-columns 7-15, with the ocean to the south.

**The mental model:** you specify a T-point, and the segment ends up enforced
on the U/V-point face right next to it.

- `index` (the odd, `2 * k + 1` supergrid value) is used *only* to pull the
  segment's own data -- the `lon`/`lat`/`mask`/forcing values come straight
  from that T-row's (or T-column's) own coordinates and wetness.
- The segment is actually enforced on a U/V-point face right next to that
  T-cell. Every T-cell has two such faces along the fixed axis -- e.g. for
  `axis="nyp"`, T-row `k` has a south face (the V-point between row `k - 1`
  and `k`) and a north face (between `k` and `k + 1`). `ocean_side` tells
  `Segment` which of those two faces is the real boundary: is the open ocean
  south or north of row `k`?
- Internally, this becomes the MOM6 `J`/`I` integer: normally it's just `k`,
  but when `ocean_side` is `"south"`/`"west"` it bumps to `k + 1` instead --
  because MOM6's own `open_boundary_impose_land_mask` is asymmetric (it masks
  the *neighbor* row for one direction, but the segment's *own* row for the
  other), so the bump is what keeps MOM6 from masking your own wet T-cell to
  land. Either way, the resulting `J`/`I` value identifies which of the
  T-cell's two adjacent faces is the boundary.

So: T-point in, correct U/V face out -- automatically. You never touch the
U/V arithmetic yourself; you just say which row/column you want the data
from and which side of it is wet.

In [4]:
interior_segment = Segment.from_hgrid(
    hgrid,
    axis="nyp",
    index=29,  # native T-row 14 -> supergrid index 2*14+1
    index_range=slice(14, 33),  # open T-columns 7-15 -> slice(2*7, 2*15+3)
    segment_name="segment_002",
    topo=topo,
    ocean_side="south",
)

print(interior_segment.mom6_obc_position_string())
interior_segment.lon.values, interior_segment.mask.values

J=15,I=16:7


(array([ 7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. , 10.5, 11. , 11.5, 12. ,
        12.5, 13. , 13.5, 14. , 14.5, 15. , 15.5, 16. ]),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

## Step 4: Building a segment from physical lon/lat instead of indices

`Segment.from_lonlat` resolves the nearest T-cell on the grid for you, so you
can think in physical coordinates instead of supergrid indices. It's exact on
a uniform rectilinear grid like this one.

This builds the same segment as Step 3 from coordinates: the T-cell centers of
row 14 sit at latitude 14.5, and those of columns 7 and 15 at longitudes 7.5
and 15.5. The position string matches Step 3's.

In [5]:
lonlat_segment = Segment.from_lonlat(
    hgrid,
    axis="nyp",
    fixed_lat=14.5,
    lon_range=(7.5, 15.5),
    segment_name="segment_003",
    topo=topo,
    ocean_side="south",
)

print(lonlat_segment.mom6_obc_position_string())
lonlat_segment.lon.values, lonlat_segment.mask.values

J=15,I=16:7


(array([ 7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. , 10.5, 11. , 11.5, 12. ,
        12.5, 13. , 13.5, 14. , 14.5, 15. , 15.5, 16. ]),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

## Step 5: Letting the bathymetry tell you which cardinal edges are open

`Segment.detect_open_cardinal_boundaries` looks at `topo.supergridmask` and
returns just the cardinal edges that actually touch ocean -- useful as a
sensible default when building a segment list for a case.
It only applies to the 4 cardinal edges -- custom/interior boundaries like
`segment_002` and `segment_003` above always have to be specified explicitly.

Our land strip covers the *entire* north edge, so `'north'` should be the one
cardinal direction missing from the result below.

In [6]:
Segment.detect_open_cardinal_boundaries(topo)

['south', 'west', 'east']

## Where to go from here

Once you have a `Segment`, the next steps are:

- `segment.regrid_velocity_tracers(...)` -- regrid raw velocity/tracer forcing
  data onto the segment and write the MOM6-ready OBC segment file.
- `segment.regrid_tides(...)` -- same, for tidal boundary forcing.
- `segment.to_spec()` / `Segment.from_spec()` -- serialize a segment's
  axis/index/index_range/orientation to a small JSON-able dict and rebuild it
  later, without holding onto the `hgrid`/`topo` objects.

Both of those need real forcing data, so they're outside the scope of this
quick geometry-focused demo -- see `regional_mom6.experiment.Experiment`'s
OBC setup for a full end-to-end example.